# 01 - Map VASA onto Controls

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import marsilea as ma
import marsilea.plotter as mp
from dynaconf import Dynaconf

# Global Scanpy settings
sc.settings.verbosity = 2               # Show progress
sc.settings.set_figure_params(
    dpi=150,                            # High-res output
    dpi_save=300,                       # High-res when saving
    format='pdf',                       # or 'pdf', 'svg'
    facecolor='white',                  # or 'none' for transparent
    frameon=False,                      # No outer frame
    vector_friendly=True,               # No rasterization warnings
    fontsize=10,                        # Base font size
    figsize=(4, 4),                     # Default figure size
    transparent=True                    # Transparent background if saving
)

mpl.rcParams.update({
    "svg.fonttype": "none",       
    "pdf.fonttype": 42,           
    "legend.fontsize": 6,
    "axes.titlesize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
})

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Load data 

In [ ]:
adata_parse = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_eec.h5ad")
adata_parse_controls = adata_parse[adata_parse.obs["condition"].isin(["Control"])]

adata_parse_controls = adata_parse_controls.raw.to_adata()

In [ ]:
# Load data
adata_vasa = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.scfates.h5ad")
adata_vasa = adata_vasa.raw.to_adata()

# Add approriate colors for the terminal states
adata_vasa.uns['term_states_fwd_colors'] = ["#ee8865","#85a7cf", "#6ac077", "#58B6D7"]

adata_vasa.obs["sample"] = adata_vasa.obs["orig.ident"]
adata_vasa.obs["annotation"] = adata_vasa.obs["cell_type_initial"]

# Standard preprocessing
adata_vasa.raw = adata_vasa.copy()
adata_vasa.layers["counts"] = adata_vasa.X.copy()

sc.pp.normalize_total(adata_vasa, target_sum=1e4)
sc.pp.log1p(adata_vasa)
adata_vasa.layers["logcounts"] = adata_vasa.X.copy()

adata_vasa.obs["annotation"] = adata_vasa.obs["annotation"].map({
    "Early X cells": "EEC Progenitors",
    "Late X cells": "X Cells",
    "Early I/K cells": "EEC Progenitors",
    "Late I/K cells": "EEC Progenitors",
    "Early EECs": "EEC Progenitors",
    "Early ECs": "EEC Progenitors",
    "Late ECs": "EC Cells",
    "D cells": "D Cells",
    "I cells": "I/N Cells",
    "K cells": "K Cells",
})

In [ ]:
# Load mapping
mapping = dict(zip(["100", "140", "186", "120", "59", "170", "165", "17"], 
                   ["X cells","D cells","X/D/I/K cells","EC/X/D/I/K cells",
                    "X/D/I/K cells","NGN3+ cells", "EC cells", "I/K cells"]))


# Transfer annotation
adata_vasa.obs["milestones_anno"] = adata_vasa.obs["milestones"].map(mapping)

In [ ]:
adata_vasa

In [ ]:
sc.pl.umap(adata_vasa, color=["annotation", "milestones"])

In [ ]:
# Subset to common genes
common_genes = adata_parse_controls.var_names.intersection(adata_vasa.var_names)
adata_parse_controls = adata_parse_controls[:, common_genes].copy()
adata_vasa = adata_vasa[:, common_genes].copy()

In [ ]:
adata = adata_vasa.concatenate(adata_parse_controls, batch_key="dataset", batch_categories=["VASA", "TF_KO_Controls"])

## Add basic QC metrics

In [ ]:
adata.raw = adata.copy()
adata.layers["counts"] = adata.X.copy()

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], inplace=True, log1p=False)

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

In [ ]:
sc.pl.violin(adata, keys=["total_counts", "pct_counts_mt"], groupby="sample", rotation=90)

## Perform cell cycle scoring

In [ ]:
cell_cycle_genes = [x.strip() for x in open('/projects/site/pred/ihb-g-deco/PUBLIC_DATA/DB/regev_lab_cell_cycle_genes.txt')]

# Split into 2 lists
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

cell_cycle_genes = [x for x in cell_cycle_genes if x in adata.var_names]

In [ ]:
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

In [ ]:
# Remove all elements under adata.uns
for key in list(adata.uns.keys()):
    del adata.uns[key]

In [ ]:
adata.raw = adata.copy()

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_control_vasa_combined_including_cystic_v3.h5ad")

## Run PCA on both datasets

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_control_vasa_combined_including_cystic_v3.h5ad")

In [ ]:
# Get highly variable genes for each dataset
sc.pp.highly_variable_genes(
    adata,
    batch_key="dataset",
    n_top_genes=2000,
    flavor="seurat_v3",
    subset=True,
    inplace=True,
)

In [ ]:
sc.pp.pca(adata)

In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["dataset", "sample", "annotation"], wspace=0.4)

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_control_vasa_combined_pca_including_cystic_v3.h5ad")